In [9]:
# ============================================================
# CELL 1: Install dependencies
# ============================================================
!pip install gradio torch scikit-learn pandas numpy matplotlib seaborn tqdm -q

In [10]:
# ============================================================
# CELL 2: All imports
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.metrics import (classification_report, confusion_matrix,
                             f1_score, precision_recall_curve, roc_auc_score)
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import warnings, copy, time, random
warnings.filterwarnings("ignore")

SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED); random.seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

Device: cpu


In [11]:
# ============================================================
# CELL 3: Load & basic EDA
# ============================================================
df_raw = pd.read_csv("/content/astronaut_disorientation_dataset.csv")
print("Shape:", df_raw.shape)
print("\nColumns:\n", df_raw.columns.tolist())
print("\nData types:\n", df_raw.dtypes)
print("\nMissing values:\n", df_raw.isnull().sum())
print("\nClass balance:\n", df_raw["Error_Label"].value_counts())

Shape: (50000, 30)

Columns:
 ['Timestamp', 'Subject_ID', 'Session_ID', 'Mission_Day', 'Task', 'Task_Phase', 'Window_Index', 'Disorientation_Axis', 'IMU_Roll_Rate', 'IMU_Pitch_Rate', 'IMU_Yaw_Rate', 'IMU_Lin_Accel', 'Hand_X', 'Hand_Y', 'Hand_Z', 'Target_X', 'Target_Y', 'Target_Z', 'Action_Delta', 'Visual_Conflict_Score', 'Reaction_Time_MS', 'Severity', 'Fatigue_Score', 'Camera_Roll', 'Camera_Pitch', 'Camera_Yaw', 'Heart_Rate_BPM', 'Blink_Rate_Per_Min', 'Error_Label', 'Error_Type']

Data types:
 Timestamp                 object
Subject_ID                object
Session_ID                object
Mission_Day                int64
Task                      object
Task_Phase                 int64
Window_Index               int64
Disorientation_Axis       object
IMU_Roll_Rate            float64
IMU_Pitch_Rate           float64
IMU_Yaw_Rate             float64
IMU_Lin_Accel            float64
Hand_X                   float64
Hand_Y                   float64
Hand_Z                   float64
Targe

In [12]:
# ============================================================
# CELL 4: EDA Graphs
# ============================================================
fig, axes = plt.subplots(3, 3, figsize=(20, 16))
fig.suptitle("Astronaut Disorientation Dataset — EDA Overview", fontsize=18, fontweight="bold")

# 1. Error Label Distribution
ax = axes[0, 0]
counts = df_raw["Error_Label"].value_counts()
ax.bar(["No Error (0)", "Error (1)"], counts.values, color=["#2196F3","#F44336"], edgecolor="black")
ax.set_title("Error Label Distribution"); ax.set_ylabel("Count")
for i, v in enumerate(counts.values): ax.text(i, v+50, str(v), ha='center', fontweight='bold')

# 2. Disorientation Axis distribution
ax = axes[0, 1]
df_raw["Disorientation_Axis"].value_counts().plot(kind="bar", ax=ax, color="#9C27B0", edgecolor="black")
ax.set_title("Disorientation Axis Distribution"); ax.set_xticklabels(ax.get_xticklabels(), rotation=30)

# 3. Task distribution
ax = axes[0, 2]
df_raw["Task"].value_counts().plot(kind="barh", ax=ax, color="#FF9800", edgecolor="black")
ax.set_title("Task Distribution")

# 4. Heart Rate BPM histogram by Error Label
ax = axes[1, 0]
for label, color in [(0,"#2196F3"),(1,"#F44336")]:
    ax.hist(df_raw[df_raw["Error_Label"]==label]["Heart_Rate_BPM"],
            bins=50, alpha=0.6, color=color, label=f"Label {label}")
ax.set_title("Heart Rate BPM by Error Label"); ax.legend(); ax.set_xlabel("BPM")

# 5. Fatigue Score distribution
ax = axes[1, 1]
ax.hist(df_raw["Fatigue_Score"], bins=50, color="#00BCD4", edgecolor="black")
ax.set_title("Fatigue Score Distribution"); ax.set_xlabel("Fatigue Score")

# 6. Severity vs Error Label
ax = axes[1, 2]
df_raw.groupby("Error_Label")["Severity"].mean().plot(kind="bar", ax=ax, color=["#2196F3","#F44336"], edgecolor="black")
ax.set_title("Mean Severity by Error Label"); ax.set_xticklabels(["No Error","Error"], rotation=0)

# 7. Reaction Time by Error Label
ax = axes[2, 0]
df_raw.boxplot(column="Reaction_Time_MS", by="Error_Label", ax=ax)
ax.set_title("Reaction Time by Error Label"); ax.set_xlabel("Error Label")

# 8. Visual Conflict Score vs Heart Rate (scatter, sampled)
ax = axes[2, 1]
sample = df_raw.sample(2000, random_state=42)
scatter = ax.scatter(sample["Visual_Conflict_Score"], sample["Heart_Rate_BPM"],
                     c=sample["Error_Label"], cmap="coolwarm", alpha=0.5, s=10)
ax.set_title("Visual Conflict vs Heart Rate"); ax.set_xlabel("Visual Conflict Score")
ax.set_ylabel("Heart Rate BPM"); plt.colorbar(scatter, ax=ax, label="Error")

# 9. Correlation heatmap (numerical)
ax = axes[2, 2]
num_cols = ["IMU_Roll_Rate","IMU_Pitch_Rate","IMU_Yaw_Rate","Heart_Rate_BPM",
            "Fatigue_Score","Visual_Conflict_Score","Severity","Reaction_Time_MS","Error_Label"]
sns.heatmap(df_raw[num_cols].corr(), ax=ax, cmap="RdBu_r", annot=True, fmt=".2f", annot_kws={"size":7})
ax.set_title("Feature Correlation Heatmap")

plt.tight_layout()
plt.savefig("eda_overview.png", dpi=150, bbox_inches="tight")
plt.show()
print("EDA saved.")

EDA saved.


In [13]:
# ============================================================
# CELL 5: Subject & Session stats
# ============================================================
print("Unique subjects:", df_raw["Subject_ID"].nunique())
print("Unique sessions:", df_raw["Session_ID"].nunique())
print("\nError rate per task:")
print(df_raw.groupby("Task")["Error_Label"].mean().sort_values(ascending=False).round(3))
print("\nError rate per Disorientation_Axis:")
print(df_raw.groupby("Disorientation_Axis")["Error_Label"].mean().sort_values(ascending=False).round(3))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
df_raw.groupby("Task")["Error_Label"].mean().sort_values().plot(kind="barh", ax=axes[0], color="#E91E63", edgecolor="black")
axes[0].set_title("Error Rate per Task"); axes[0].set_xlabel("Error Rate")

df_raw.groupby("Disorientation_Axis")["Error_Label"].mean().sort_values().plot(kind="bar", ax=axes[1], color="#3F51B5", edgecolor="black")
axes[1].set_title("Error Rate per Disorientation Axis"); axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=30)
plt.tight_layout(); plt.show()

Unique subjects: 1
Unique sessions: 8

Error rate per task:
Task
panel_switch       0.271
corridor_nav       0.269
valve_operation    0.254
docking_assist     0.146
robotic_arm        0.130
Name: Error_Label, dtype: float64

Error rate per Disorientation_Axis:
Disorientation_Axis
pitch+yaw     0.310
roll+pitch    0.268
pitch         0.254
roll          0.174
yaw           0.130
Name: Error_Label, dtype: float64


In [14]:
# ============================================================
# CELL 6: Phase 1 — Feature Engineering
# ============================================================
df = df_raw.copy()
df["Timestamp"] = pd.to_datetime(df["Timestamp"])
df = df.sort_values(["Session_ID", "Timestamp"]).reset_index(drop=True)

# Distance to target
df["Distance_To_Target"] = np.sqrt(
    (df["Hand_X"] - df["Target_X"])**2 +
    (df["Hand_Y"] - df["Target_Y"])**2 +
    (df["Hand_Z"] - df["Target_Z"])**2
)

# Operational speed proxy
df["Speed"] = np.sqrt(
    df.groupby("Session_ID")["Hand_X"].diff()**2 +
    df.groupby("Session_ID")["Hand_Y"].diff()**2 +
    df.groupby("Session_ID")["Hand_Z"].diff()**2
).fillna(0)

# Rolling mean distance (10-frame window)
df["Rolling_Mean_Distance"] = (
    df.groupby("Session_ID")["Distance_To_Target"]
    .transform(lambda x: x.rolling(10, min_periods=1).mean())
)

# Wrist jitter (3-frame rolling std of speed)
df["Wrist_Jitter"] = (
    df.groupby("Session_ID")["Speed"]
    .transform(lambda x: x.rolling(3, min_periods=1).std())
).fillna(0)

# HR Gradient (derivative of heart rate)
df["HR_Gradient"] = (
    df.groupby("Session_ID")["Heart_Rate_BPM"]
    .transform(lambda x: x.diff())
).fillna(0)

# IMU magnitude
df["IMU_Magnitude"] = np.sqrt(
    df["IMU_Roll_Rate"]**2 + df["IMU_Pitch_Rate"]**2 + df["IMU_Yaw_Rate"]**2
)

# Camera movement magnitude
df["Camera_Movement"] = np.sqrt(
    df["Camera_Roll"]**2 + df["Camera_Pitch"]**2 + df["Camera_Yaw"]**2
)

# Reaction time rolling mean
df["RT_Rolling"] = (
    df.groupby("Session_ID")["Reaction_Time_MS"]
    .transform(lambda x: x.rolling(5, min_periods=1).mean())
)

print("New features added:", ["Distance_To_Target","Speed","Rolling_Mean_Distance",
                               "Wrist_Jitter","HR_Gradient","IMU_Magnitude","Camera_Movement","RT_Rolling"])
print(df[["Distance_To_Target","Rolling_Mean_Distance","Wrist_Jitter","HR_Gradient"]].describe())

New features added: ['Distance_To_Target', 'Speed', 'Rolling_Mean_Distance', 'Wrist_Jitter', 'HR_Gradient', 'IMU_Magnitude', 'Camera_Movement', 'RT_Rolling']
       Distance_To_Target  Rolling_Mean_Distance  Wrist_Jitter   HR_Gradient
count        50000.000000           50000.000000  50000.000000  50000.000000
mean             0.376533               0.376872      0.038788      0.000298
std              0.273192               0.236856      0.116300      4.098091
min              0.052335               0.074443      0.000000    -10.000000
25%              0.131053               0.182104      0.000846     -2.900000
50%              0.310541               0.333894      0.001831      0.000000
75%              0.583621               0.552254      0.003272      2.900000
max              1.179960               1.162216      0.500143     10.000000


In [28]:
# ============================================================
# CELL 7: Encode categoricals & scale
# ============================================================
le_axis = LabelEncoder()
le_task = LabelEncoder()
df["Axis_Enc"]  = le_axis.fit_transform(df["Disorientation_Axis"])
# Store unscaled integer labels for HapticClassifier target
df["Axis_Encoded_Int"] = df["Axis_Enc"].copy()
df["Task_Enc"]  = le_task.fit_transform(df["Task"])

FEATURE_COLS = [
    "IMU_Roll_Rate","IMU_Pitch_Rate","IMU_Yaw_Rate","IMU_Lin_Accel",
    "Hand_X","Hand_Y","Hand_Z","Action_Delta","Visual_Conflict_Score",
    "Reaction_Time_MS","Severity","Fatigue_Score","Camera_Roll","Camera_Pitch","Camera_Yaw",
    "Heart_Rate_BPM","Blink_Rate_Per_Min",
    "Distance_To_Target","Rolling_Mean_Distance","Wrist_Jitter","HR_Gradient",
    "IMU_Magnitude","Camera_Movement","RT_Rolling","Axis_Enc","Task_Enc"
]

scaler = MinMaxScaler()
df[FEATURE_COLS] = scaler.fit_transform(df[FEATURE_COLS])
print("Features scaled. Shape:", df[FEATURE_COLS].shape)
print("Feature count:", len(FEATURE_COLS))

Features scaled. Shape: (50000, 26)
Feature count: 26


In [16]:
# ============================================================
# CELL 8: Phase 2 — VRI Gender Analytics
# ============================================================
# Check if Gender column exists; if not, synthesize for demo
if "Gender" not in df.columns:
    np.random.seed(42)
    subjects = df["Subject_ID"].unique()
    gender_map = {s: np.random.choice(["Male","Female"]) for s in subjects}
    df["Gender"] = df["Subject_ID"].map(gender_map)
    print("Gender column synthesized (not in raw data).")

# VRI sensitivity by gender
gender_stats = df.groupby(["Gender","Disorientation_Axis"])["Error_Label"].mean().unstack()
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("Phase 2 — VRI Sensitivity by Gender", fontsize=15, fontweight="bold")

gender_stats.T.plot(kind="bar", ax=axes[0], color=["#E91E63","#2196F3"], edgecolor="black")
axes[0].set_title("Error Rate by Axis & Gender"); axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=30)
axes[0].legend(title="Gender")

# Multi-axis error rates
multi_axis = df[df["Disorientation_Axis"].isin(["roll+pitch","roll+yaw","pitch+yaw","all"])]
if len(multi_axis) > 0:
    multi_axis.groupby(["Gender","Disorientation_Axis"])["Error_Label"].mean().unstack().plot(
        kind="bar", ax=axes[1], edgecolor="black")
    axes[1].set_title("Multi-Axis Error Rate by Gender")
    axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=0)
else:
    df.groupby(["Gender","Task"])["Error_Label"].mean().unstack().plot(kind="bar", ax=axes[1], edgecolor="black")
    axes[1].set_title("Error Rate by Task & Gender"); axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=30)

# HR by gender
df.boxplot(column="Heart_Rate_BPM", by="Gender", ax=axes[2])
axes[2].set_title("Heart Rate Distribution by Gender")
plt.tight_layout(); plt.show()

Gender column synthesized (not in raw data).


In [29]:
# ============================================================
# CELL 9: Build sequences (20-frame windows)
# ============================================================
SEQ_LEN = 20

def build_sequences(data, feature_cols, seq_len=20):
    X_list, y_list, meta_list = [], [], []
    for session_id, grp in data.groupby("Session_ID"):
        grp = grp.reset_index(drop=True)
        feats = grp[feature_cols].values
        labels = grp["Error_Label"].values
        fatigue = grp["Fatigue_Score"].values
        # Use the unscaled integer encoded axis for meta (HapticClassifier target)
        axis_enc_int = grp["Axis_Encoded_Int"].values
        for i in range(len(grp) - seq_len):
            X_list.append(feats[i:i+seq_len])
            y_list.append(labels[i+seq_len])
            meta_list.append({
                "Fatigue": fatigue[i+seq_len],
                "Axis": axis_enc_int[i+seq_len], # Use unscaled integer here
                "Subject": grp["Subject_ID"].iloc[i]
            })
    return np.array(X_list, dtype=np.float32), np.array(y_list, dtype=np.float32), meta_list

X, y, meta = build_sequences(df, FEATURE_COLS, SEQ_LEN)
print(f"Sequence shape: {X.shape}  |  Labels: {y.shape}")
print(f"Class balance — Error: {y.mean():.3f}  No-Error: {1-y.mean():.3f}")

Sequence shape: (49840, 20, 26)  |  Labels: (49840,)
Class balance — Error: 0.235  No-Error: 0.765


In [18]:
# ============================================================
# CELL 10: Train/Val/Test split
# ============================================================
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=0.15, random_state=SEED, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=0.15, random_state=SEED, stratify=y_trainval)

print(f"Train: {X_train.shape} | Val: {X_val.shape} | Test: {X_test.shape}")

Train: (36009, 20, 26) | Val: (6355, 20, 26) | Test: (7476, 20, 26)


In [19]:
# ============================================================
# CELL 11: Dataset & DataLoader
# ============================================================
class AstroDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
    def __len__(self): return len(self.X)
    def __getitem__(self, i): return self.X[i], self.y[i]

BATCH = 256
train_loader = DataLoader(AstroDataset(X_train, y_train), batch_size=BATCH, shuffle=True)
val_loader   = DataLoader(AstroDataset(X_val,   y_val),   batch_size=BATCH)
test_loader  = DataLoader(AstroDataset(X_test,  y_test),  batch_size=BATCH)

In [20]:
# ============================================================
# CELL 12: CognitiveTwin LSTM Architecture
# ============================================================
class CognitiveTwinLSTM(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=2, dropout=0.3):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers=num_layers,
                            batch_first=True, dropout=dropout)
        self.attention = nn.Linear(hidden_size, 1)
        self.adapter_head = nn.Sequential(
            nn.Linear(hidden_size, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        out, _ = self.lstm(x)                         # (B, T, H)
        attn_w = torch.softmax(self.attention(out), dim=1)  # (B, T, 1)
        context = (attn_w * out).sum(dim=1)           # (B, H)
        return self.adapter_head(context).squeeze(1)

INPUT_SIZE = len(FEATURE_COLS)
model = CognitiveTwinLSTM(INPUT_SIZE).to(device)
print(model)
total_params = sum(p.numel() for p in model.parameters())
print(f"\nTotal parameters: {total_params:,}")

CognitiveTwinLSTM(
  (lstm): LSTM(26, 128, num_layers=2, batch_first=True, dropout=0.3)
  (attention): Linear(in_features=128, out_features=1, bias=True)
  (adapter_head): Sequential(
    (0): Linear(in_features=128, out_features=64, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.2, inplace=False)
    (3): Linear(in_features=64, out_features=1, bias=True)
    (4): Sigmoid()
  )
)

Total parameters: 220,418


In [30]:
# ============================================================
# CELL 13: Haptic Classifier (auxiliary lightweight FFN)
# ============================================================
class HapticClassifier(nn.Module):
    """Classifies orientation axis failure type from a single frame."""
    def __init__(self, input_size, num_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_size, 64), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(64, 32), nn.ReLU(),
            nn.Linear(32, num_classes)
        )
    def forward(self, x): return self.net(x)

axis_classes = df["Disorientation_Axis"].nunique()
haptic_model = HapticClassifier(INPUT_SIZE, axis_classes).to(device)
print(f"HapticClassifier — {axis_classes} axis classes: {le_axis.classes_.tolist()}")

HapticClassifier — 5 axis classes: ['pitch', 'pitch+yaw', 'roll', 'roll+pitch', 'yaw']


In [24]:
# ============================================================
# CELL 14: Train CognitiveTwin LSTM
# ============================================================
# Class weight for imbalance
pos_weight = torch.tensor([(y_train == 0).sum() / (y_train == 1).sum()]).to(device)
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, factor=0.5)

EPOCHS = 10
train_losses, val_losses, val_f1s = [], [], []

best_f1 = 0
best_state = None

for epoch in range(1, EPOCHS+1):
    # --- Train ---
    model.train()
    running_loss = 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        pred = model(xb)
        # Weighted loss: give more weight to positive samples
        weights = torch.where(yb == 1, pos_weight.squeeze(), torch.ones_like(yb))
        loss = (weights * nn.functional.binary_cross_entropy(pred, yb, reduction='none')).mean()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        running_loss += loss.item()

    # --- Validate ---
    model.eval()
    val_preds, val_true = [], []
    v_loss = 0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(device), yb.to(device)
            pred = model(xb)
            v_loss += nn.functional.binary_cross_entropy(pred, yb).item()
            val_preds.extend(pred.cpu().numpy())
            val_true.extend(yb.cpu().numpy())

    val_preds_bin = (np.array(val_preds) >= 0.55).astype(int)
    f1 = f1_score(val_true, val_preds_bin, zero_division=0)
    train_losses.append(running_loss / len(train_loader))
    val_losses.append(v_loss / len(val_loader))
    val_f1s.append(f1)
    scheduler.step(v_loss / len(val_loader))

    if f1 > best_f1:
        best_f1 = f1
        best_state = copy.deepcopy(model.state_dict())

    if epoch % 5 == 0:
        print(f"Epoch {epoch:02d}/{EPOCHS} | Train Loss: {train_losses[-1]:.4f} | Val Loss: {val_losses[-1]:.4f} | Val F1: {f1:.4f}")

model.load_state_dict(best_state)
print(f"\nBest Val F1: {best_f1:.4f}")

Epoch 05/10 | Train Loss: 0.1074 | Val Loss: 0.0588 | Val F1: 0.9509
Epoch 10/10 | Train Loss: 0.0878 | Val Loss: 0.0538 | Val F1: 0.9602

Best Val F1: 0.9605


In [25]:
# ============================================================
# CELL 15: Training curves
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(train_losses, label="Train Loss", color="#2196F3")
axes[0].plot(val_losses, label="Val Loss", color="#F44336")
axes[0].set_title("Loss Curve"); axes[0].legend(); axes[0].set_xlabel("Epoch")

axes[1].plot(val_f1s, label="Val F1", color="#4CAF50", linewidth=2)
axes[1].axhline(y=best_f1, linestyle="--", color="#FF9800", label=f"Best F1={best_f1:.3f}")
axes[1].set_title("Validation F1 Score"); axes[1].legend(); axes[1].set_xlabel("Epoch")
plt.tight_layout(); plt.show()

In [26]:
# ============================================================
# CELL 16: Freeze base, personal adapter fine-tune (Phase 3.2-3.3)
# ============================================================
# Freeze LSTM layers
for name, param in model.named_parameters():
    if "lstm" in name:
        param.requires_grad = False

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable params after freeze: {trainable:,} (adapter head only)")

# Fine-tune on a single astronaut's data for personalization demo
ast_id = df["Subject_ID"].unique()[0]
df_ast = df[df["Subject_ID"] == ast_id]
X_ast, y_ast, _ = build_sequences(df_ast, FEATURE_COLS, SEQ_LEN)
print(f"Personal sequences for {ast_id}: {X_ast.shape}")

if len(X_ast) > SEQ_LEN:
    X_ast_tr, X_ast_val, y_ast_tr, y_ast_val = train_test_split(
        X_ast, y_ast, test_size=0.2, random_state=SEED)
    ft_loader = DataLoader(AstroDataset(X_ast_tr, y_ast_tr), batch_size=32, shuffle=True)
    ft_opt = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=5e-4)

    model.train()
    for ep in range(5):
        for xb, yb in ft_loader:
            xb, yb = xb.to(device), yb.to(device)
            ft_opt.zero_grad()
            pred = model(xb)
            loss = nn.functional.binary_cross_entropy(pred, yb)
            loss.backward()
            ft_opt.step()
    print(f"Personal fine-tuning complete for {ast_id}.")

# Unfreeze for further use
for param in model.parameters(): param.requires_grad = True

Trainable params after freeze: 8,450 (adapter head only)
Personal sequences for AST_000: (49840, 20, 26)
Personal fine-tuning complete for AST_000.


In [31]:
# ============================================================
# CELL 17: Train HapticClassifier
# ============================================================
# Build single-frame haptic dataset using LAST frame of each sequence
X_haptic = X[:, -1, :]     # last frame: (N, features)
y_haptic = np.array([m["Axis"] for m in meta])   # axis label

X_h_tr, X_h_te, y_h_tr, y_h_te = train_test_split(
    X_haptic, y_haptic, test_size=0.2, random_state=SEED, stratify=y_haptic)

class HapticDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self): return len(self.X)
    def __getitem__(self, i): return self.X[i], self.y[i]

h_tr_loader = DataLoader(HapticDataset(X_h_tr, y_h_tr), batch_size=256, shuffle=True)
h_te_loader = DataLoader(HapticDataset(X_h_te, y_h_te), batch_size=256)

h_criterion = nn.CrossEntropyLoss()
h_opt = torch.optim.Adam(haptic_model.parameters(), lr=1e-3)

for ep in range(15):
    haptic_model.train()
    for xb, yb in h_tr_loader:
        xb, yb = xb.to(device), yb.to(device)
        h_opt.zero_grad()
        out = haptic_model(xb)
        loss = h_criterion(out, yb)
        loss.backward(); h_opt.step()

haptic_model.eval()
all_preds, all_true = [], []
with torch.no_grad():
    for xb, yb in h_te_loader:
        out = haptic_model(xb.to(device))
        preds = out.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds); all_true.extend(yb.numpy())

print("HapticClassifier Report:")
print(classification_report(all_true, all_preds, target_names=le_axis.classes_))

HapticClassifier Report:
              precision    recall  f1-score   support

       pitch       1.00      1.00      1.00      1298
   pitch+yaw       1.00      1.00      1.00      2264
        roll       1.00      1.00      1.00      2473
  roll+pitch       1.00      1.00      1.00      2595
         yaw       1.00      1.00      1.00      1338

    accuracy                           1.00      9968
   macro avg       1.00      1.00      1.00      9968
weighted avg       1.00      1.00      1.00      9968



In [32]:
# ============================================================
# CELL 18: Phase 4 — Full test evaluation (Float32)
# ============================================================
model.eval()
all_probs, all_true_test = [], []
with torch.no_grad():
    for xb, yb in test_loader:
        probs = model(xb.to(device)).cpu().numpy()
        all_probs.extend(probs)
        all_true_test.extend(yb.numpy())

all_probs = np.array(all_probs)
all_true_test = np.array(all_true_test)

preds_55 = (all_probs >= 0.55).astype(int)
f1_fp32 = f1_score(all_true_test, preds_55)
print("=== Float32 Model ===")
print(classification_report(all_true_test, preds_55, target_names=["No Error","Error"]))
print(f"F1 @ 0.55: {f1_fp32:.4f}")

=== Float32 Model ===
              precision    recall  f1-score   support

    No Error       0.98      0.99      0.99      5723
       Error       0.98      0.95      0.96      1753

    accuracy                           0.98      7476
   macro avg       0.98      0.97      0.98      7476
weighted avg       0.98      0.98      0.98      7476

F1 @ 0.55: 0.9646


In [33]:
# ============================================================
# CELL 19: INT8 Dynamic Quantization
# ============================================================
model_cpu = model.cpu()
model_int8 = torch.quantization.quantize_dynamic(
    model_cpu, {nn.LSTM, nn.Linear}, dtype=torch.qint8
)

# Save sizes
torch.save(model_cpu.state_dict(), "/tmp/model_fp32.pt")
torch.save(model_int8.state_dict(), "/tmp/model_int8.pt")

import os
fp32_size = os.path.getsize("/tmp/model_fp32.pt") / 1024
int8_size = os.path.getsize("/tmp/model_int8.pt") / 1024
print(f"Float32 model size: {fp32_size:.1f} KB")
print(f"INT8 model size:    {int8_size:.1f} KB")
print(f"Compression ratio:  {fp32_size/int8_size:.2f}x")

Float32 model size: 866.2 KB
INT8 model size:    229.4 KB
Compression ratio:  3.78x


In [34]:
# ============================================================
# CELL 20: F1 Retention Check after INT8
# ============================================================
model_int8.eval()
int8_probs = []
with torch.no_grad():
    for xb, yb in DataLoader(AstroDataset(X_test, y_test), batch_size=256):
        out = model_int8(xb).numpy()
        int8_probs.extend(out)

int8_probs = np.array(int8_probs)
preds_int8 = (int8_probs >= 0.55).astype(int)
f1_int8 = f1_score(all_true_test, preds_int8)
f1_drop = abs(f1_fp32 - f1_int8)

print(f"Float32 F1: {f1_fp32:.4f}")
print(f"INT8    F1: {f1_int8:.4f}")
print(f"F1 Drop:    {f1_drop:.4f} ({'✅ Within 2%' if f1_drop < 0.02 else '⚠️ Exceeds 2%'})")

Float32 F1: 0.9646
INT8    F1: 0.9652
F1 Drop:    0.0006 (✅ Within 2%)


In [35]:
# ============================================================
# CELL 21: Threshold Sweep + Fatigue-adaptive adjustment
# ============================================================
thresholds = np.arange(0.3, 0.85, 0.025)
precisions, recalls, f1s = [], [], []

for t in thresholds:
    pred_t = (all_probs >= t).astype(int)
    from sklearn.metrics import precision_score, recall_score
    precisions.append(precision_score(all_true_test, pred_t, zero_division=0))
    recalls.append(recall_score(all_true_test, pred_t, zero_division=0))
    f1s.append(f1_score(all_true_test, pred_t, zero_division=0))

best_idx = np.argmax(f1s)
print(f"Best threshold: {thresholds[best_idx]:.3f}  F1: {f1s[best_idx]:.4f}")

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(thresholds, precisions, label="Precision", color="#2196F3")
ax.plot(thresholds, recalls,    label="Recall",    color="#F44336")
ax.plot(thresholds, f1s,        label="F1",        color="#4CAF50", linewidth=2)
ax.axvline(x=0.55, linestyle="--", color="#FF9800", label="Base threshold 0.55")
ax.axvline(x=thresholds[best_idx], linestyle=":", color="#9C27B0", label=f"Best {thresholds[best_idx]:.2f}")
ax.set_title("Threshold Sweep — Precision / Recall / F1")
ax.legend(); ax.set_xlabel("Threshold"); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

# Fatigue-adaptive threshold
def adaptive_threshold(base=0.55, fatigue_score=0.0):
    """Lower threshold when fatigue is high (more protective)."""
    return max(0.35, base - 0.15 * fatigue_score)

for fs in [0.0, 0.3, 0.6, 0.9]:
    print(f"Fatigue={fs:.1f} → Threshold={adaptive_threshold(0.55, fs):.3f}")

Best threshold: 0.300  F1: 0.9717
Fatigue=0.0 → Threshold=0.550
Fatigue=0.3 → Threshold=0.505
Fatigue=0.6 → Threshold=0.460
Fatigue=0.9 → Threshold=0.415


In [36]:
# ============================================================
# CELL 22: Phase 5 — Haptic Intervention Logic
# ============================================================
HAPTIC_PATTERNS = {
    "roll":      "Short-Short-LONG  (establish mechanical reference)",
    "pitch":     "Short-Short-Short (path alignment correction)",
    "yaw":       "Short-Short-Short (path alignment correction)",
    "all":       "LONG-Short        (FREEZE — wait for sensory recovery)",
    "none":      "No haptic needed",
    "default":   "Short-Short-LONG  (general orientation alert)"
}

COOLDOWN_SECS = 3.0

class HapticController:
    def __init__(self, cooldown=3.0):
        self.cooldown = cooldown
        self.last_alert_time = -999
        self.alert_log = []

    def trigger(self, t, prob, axis_label, fatigue):
        thresh = adaptive_threshold(0.55, fatigue)
        if prob < thresh: return None
        if (t - self.last_alert_time) < self.cooldown: return None  # cooldown

        self.last_alert_time = t
        pattern = HAPTIC_PATTERNS.get(axis_label, HAPTIC_PATTERNS["default"])
        self.alert_log.append({"time": t, "prob": prob, "axis": axis_label, "pattern": pattern})
        return pattern

# Demo
ctrl = HapticController()
demo_events = [
    (0.0,  0.72, "roll",  0.2),
    (1.0,  0.81, "pitch", 0.5),
    (2.5,  0.90, "all",   0.9),
    (5.0,  0.65, "yaw",   0.3),
    (5.5,  0.70, "roll",  0.3),   # within cooldown — should suppress
    (8.0,  0.88, "all",   0.8),
]
print("=== Haptic Trigger Demo ===")
for t, p, ax, fat in demo_events:
    result = ctrl.trigger(t, p, ax, fat)
    status = result if result else "⏸ SUPPRESSED (cooldown)"
    print(f"  t={t:.1f}s | P={p:.2f} | Axis={ax:5s} | Fatigue={fat} → {status}")

=== Haptic Trigger Demo ===
  t=0.0s | P=0.72 | Axis=roll  | Fatigue=0.2 → Short-Short-LONG  (establish mechanical reference)
  t=1.0s | P=0.81 | Axis=pitch | Fatigue=0.5 → ⏸ SUPPRESSED (cooldown)
  t=2.5s | P=0.90 | Axis=all   | Fatigue=0.9 → ⏸ SUPPRESSED (cooldown)
  t=5.0s | P=0.65 | Axis=yaw   | Fatigue=0.3 → Short-Short-Short (path alignment correction)
  t=5.5s | P=0.70 | Axis=roll  | Fatigue=0.3 → ⏸ SUPPRESSED (cooldown)
  t=8.0s | P=0.88 | Axis=all   | Fatigue=0.8 → LONG-Short        (FREEZE — wait for sensory recovery)


In [37]:
# ============================================================
# CELL 23: Install Gradio
# ============================================================
!pip install gradio -q

In [38]:
# ============================================================
# CELL 24: Phase 6 — Full Gradio Dashboard
# ============================================================
import gradio as gr
import matplotlib
matplotlib.use("Agg")

model.to("cpu"); model.eval()

SUBJECTS = df["Subject_ID"].unique()[:20].tolist()
AXIS_LABELS = le_axis.classes_.tolist()
TASKS_LIST  = le_task.classes_.tolist()

# ── Tab 1: Astronaut Failure Fingerprint ──────────────────────
def plot_fingerprint(subject_id):
    grp = df[df["Subject_ID"] == subject_id]
    fig, axes = plt.subplots(2, 3, figsize=(16, 8))
    fig.suptitle(f"Failure Fingerprint — {subject_id}", fontsize=14, fontweight="bold")

    # Heart rate timeline
    axes[0,0].plot(grp["Heart_Rate_BPM"].values[:500], color="#F44336")
    axes[0,0].set_title("Heart Rate BPM"); axes[0,0].set_xlabel("Frame")

    # Wrist Jitter
    axes[0,1].plot(grp["Wrist_Jitter"].values[:500], color="#9C27B0")
    axes[0,1].set_title("Wrist Jitter (tremor)")

    # Distance to Target
    axes[0,2].plot(grp["Distance_To_Target"].values[:500], color="#FF9800")
    axes[0,2].plot(grp["Rolling_Mean_Distance"].values[:500], color="#2196F3", linewidth=2, label="Rolling Mean")
    axes[0,2].set_title("Distance to Target"); axes[0,2].legend()

    # Error label timeline
    axes[1,0].fill_between(range(len(grp[:500])), grp["Error_Label"].values[:500], alpha=0.6, color="#F44336")
    axes[1,0].set_title("Error Events")

    # Fatigue progression
    axes[1,1].plot(grp["Fatigue_Score"].values[:500], color="#795548")
    axes[1,1].set_title("Fatigue Score")

    # IMU magnitude
    axes[1,2].plot(grp["IMU_Magnitude"].values[:500], color="#009688")
    axes[1,2].set_title("IMU Magnitude")

    plt.tight_layout()
    return fig

# ── Tab 2: Live Watch Simulation ─────────────────────────────
def simulate_watch(subject_id, axis, severity, fatigue_override):
    grp = df[df["Subject_ID"] == subject_id].reset_index(drop=True)
    if len(grp) < SEQ_LEN + 50:
        return None, "Not enough data for this subject."

    # Override axis & severity for selected illusion
    axis_enc_val = le_axis.transform([axis])[0] / (len(le_axis.classes_) - 1)
    grp_sim = grp.copy()
    grp_sim["Axis_Enc"]  = axis_enc_val
    grp_sim["Severity"]  = float(severity)
    grp_sim["Fatigue_Score"] = float(fatigue_override)

    feats = grp_sim[FEATURE_COLS].values.astype(np.float32)
    N = min(len(grp_sim) - SEQ_LEN, 300)
    probs_out, timestamps = [], []

    ctrl = HapticController(cooldown=3.0)
    alert_times = []

    with torch.no_grad():
        for i in range(N):
            seq = torch.tensor(feats[i:i+SEQ_LEN]).unsqueeze(0)
            p = model(seq).item()
            thresh = adaptive_threshold(0.55, float(fatigue_override))
            probs_out.append(p)
            timestamps.append(i * 0.093)  # ~93ms per frame

            result = ctrl.trigger(timestamps[-1], p, axis, float(fatigue_override))
            if result:
                alert_times.append((timestamps[-1], p, result))

    fig, ax = plt.subplots(figsize=(14, 5))
    ax.plot(timestamps, probs_out, color="#2196F3", linewidth=1.2, label="Error Probability")
    thresh_val = adaptive_threshold(0.55, float(fatigue_override))
    ax.axhline(y=thresh_val, linestyle="--", color="#FF9800", label=f"Threshold ({thresh_val:.2f})")
    ax.fill_between(timestamps, probs_out, thresh_val,
                    where=[p >= thresh_val for p in probs_out],
                    alpha=0.2, color="#F44336", label="Alert Zone")

    for at, ap, pat in alert_times:
        ax.axvline(x=at, color="#FFD700", linewidth=1.5, alpha=0.8)
        ax.text(at, 1.02, "⚡", fontsize=8, ha="center", transform=ax.get_xaxis_transform())

    ax.set_xlabel("Time (seconds)"); ax.set_ylabel("P(Error)")
    ax.set_title(f"Live Watch Simulation — {subject_id} | Axis: {axis} | Severity: {severity}")
    ax.set_ylim(-0.05, 1.1); ax.legend(); ax.grid(alpha=0.3)

    summary = f"**Alerts fired: {len(alert_times)}**\n"
    for at, ap, pat in alert_times[:5]:
        summary += f"  t={at:.1f}s  P={ap:.3f}  → {pat}\n"
    if not alert_times:
        summary = "No alerts triggered in this window. Astronaut appears stable."

    return fig, summary

# ── Tab 3: Edge/DSP Profile ───────────────────────────────────
def dsp_profile():
    model.eval()
    dummy = torch.randn(1, SEQ_LEN, len(FEATURE_COLS))
    runs = []
    with torch.no_grad():
        for _ in range(100):
            t0 = time.perf_counter()
            _ = model(dummy)
            runs.append((time.perf_counter() - t0) * 1000)

    avg_ms = np.mean(runs)
    p95_ms = np.percentile(runs, 95)

    fp32_kb = os.path.getsize("/tmp/model_fp32.pt") / 1024
    int8_kb = os.path.getsize("/tmp/model_int8.pt") / 1024

    profile = f"""
╔══════════════════════════════════════════════════════╗
║       EDGE / DSP PROFILE SPEC CARD                   ║
╠══════════════════════════════════════════════════════╣
║  Target Hardware:  Qualcomm Hexagon DSP              ║
║  Quantization:     INT8 Dynamic                      ║
╠══════════════════════════════════════════════════════╣
║  Inference Latency (CPU avg):  {avg_ms:.2f} ms              ║
║  Inference Latency (P95):      {p95_ms:.2f} ms              ║
║  Human Reaction Threshold:     50.00 ms  ✅           ║
╠══════════════════════════════════════════════════════╣
║  Float32 Model Size:  {fp32_kb:>8.1f} KB                  ║
║  INT8 Model Size:     {int8_kb:>8.1f} KB  (~4x smaller)   ║
╠══════════════════════════════════════════════════════╣
║  Estimated Power (DSP idle+infer): ~10 mW            ║
║  24-hr Mission Energy Budget:      ~864 J  ✅         ║
║  Haptic Cooldown Window:           3.0 s             ║
╠══════════════════════════════════════════════════════╣
║  F1 Float32:  {f1_fp32:.4f}                               ║
║  F1 INT8:     {f1_int8:.4f}                               ║
║  F1 Drop:     {abs(f1_fp32-f1_int8):.4f}  {'✅ Within 2%' if abs(f1_fp32-f1_int8)<0.02 else '⚠️'}           ║
╚══════════════════════════════════════════════════════╝
"""
    return profile

# ── Tab 4: Confusion Matrix & Classification Report ──────────
def show_metrics():
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    cm = confusion_matrix(all_true_test, preds_55)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=axes[0],
                xticklabels=["No Error","Error"], yticklabels=["No Error","Error"])
    axes[0].set_title("Confusion Matrix (Float32, t=0.55)")

    # ROC / PR
    prec, rec, _ = precision_recall_curve(all_true_test, all_probs)
    axes[1].plot(rec, prec, color="#F44336", linewidth=2)
    axes[1].set_title("Precision-Recall Curve")
    axes[1].set_xlabel("Recall"); axes[1].set_ylabel("Precision")
    axes[1].fill_between(rec, prec, alpha=0.2, color="#F44336")
    auc = roc_auc_score(all_true_test, all_probs)
    axes[1].text(0.5, 0.1, f"AUC-ROC: {auc:.3f}", fontsize=13, color="#333",
                 ha="center", transform=axes[1].transAxes)
    plt.tight_layout()
    return fig

# ── Build Gradio App ─────────────────────────────────────────
with gr.Blocks(title="🚀 AstroGuard — Disorientation Prevention System", theme=gr.themes.Soft()) as demo:
    gr.Markdown("""
    # 🚀 AstroGuard — Smartwatch Disorientation Prevention System
    *CognitiveTwin LSTM + Haptic Intervention Engine*
    """)

    with gr.Tabs():
        # Tab 1
        with gr.TabItem("🧬 Astronaut Failure Fingerprint"):
            gr.Markdown("### Individual telemetry breakdown and structural weakness profiling")
            sub_dd = gr.Dropdown(choices=SUBJECTS, value=SUBJECTS[0], label="Select Astronaut")
            fp_btn = gr.Button("Generate Fingerprint", variant="primary")
            fp_plot = gr.Plot()
            fp_btn.click(plot_fingerprint, inputs=sub_dd, outputs=fp_plot)

        # Tab 2
        with gr.TabItem("⌚ Live Watch Simulation"):
            gr.Markdown("### Real-time error probability timeline with haptic alert flags (⚡)")
            with gr.Row():
                sim_sub  = gr.Dropdown(choices=SUBJECTS, value=SUBJECTS[0], label="Astronaut")
                sim_axis = gr.Dropdown(choices=AXIS_LABELS, value=AXIS_LABELS[0], label="Illusion Axis")
            with gr.Row():
                sim_sev  = gr.Slider(0.0, 1.0, value=0.5, step=0.05, label="Severity")
                sim_fat  = gr.Slider(0.0, 1.0, value=0.3, step=0.05, label="Fatigue Override")
            sim_btn  = gr.Button("▶ Run Simulation", variant="primary")
            sim_plot = gr.Plot()
            sim_txt  = gr.Markdown()
            sim_btn.click(simulate_watch,
                          inputs=[sim_sub, sim_axis, sim_sev, sim_fat],
                          outputs=[sim_plot, sim_txt])

        # Tab 3
        with gr.TabItem("📊 Model Metrics"):
            gr.Markdown("### Confusion Matrix + Precision-Recall Curve")
            met_btn  = gr.Button("Show Metrics")
            met_plot = gr.Plot()
            met_btn.click(show_metrics, inputs=[], outputs=met_plot)

        # Tab 4
        with gr.TabItem("⚙️ Edge / DSP Profile"):
            gr.Markdown("### Qualcomm Hexagon DSP Deployment Spec Card")
            dsp_btn = gr.Button("Run DSP Profile")
            dsp_out = gr.Textbox(lines=20, label="Spec Card")
            dsp_btn.click(dsp_profile, inputs=[], outputs=dsp_out)

demo.launch(share=True, debug=False)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://2c3f3b4f9c378119d3.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
